<img style="float: left; margin: 30px 15px 15px 15px;" src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="500" height="250" /> 
    
    
# <font color='navy'> Modelo de Puntuación Crediticia PyMES

<font color='black'>

- Diego Lozoya Morales
- Ivanna herrera Ibarra
- Ana Sofía Hinojosa Bale
- Arantza Gomez Haro Gamboa
- Javier Alejandro Fajardo López
- Luis Fernando Márquez Bañuelos

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import percentileofscore

In [2]:
df_tickers = pd.read_csv('tickers_bmv_yahoo.csv')
all_tickers = df_tickers['YAHOO_TICKER'].tolist()

## <font color='cornflowerblue'> Financials

In [3]:
def get_financials(ticker:str) -> pd.DataFrame:
    bs = yf.Ticker(ticker).balance_sheet.iloc[:, :1]
    ist = yf.Ticker(ticker).income_stmt.iloc[:, :1]
    cf = yf.Ticker(ticker).cash_flow.iloc[:, :1]

    return bs, ist, cf

## <font color='cornflowerblue'> Variables

In [4]:
def model_variables(ticker: str, bs: pd.DataFrame, ist: pd.DataFrame) -> pd.DataFrame:
    ebit_margin = ist.loc['EBIT'] / ist.loc['Total Revenue']
    cash_ratio = bs.loc['Cash And Cash Equivalents'] / bs.loc['Current Liabilities']
    ROA = ist.loc['Net Income'] / bs.loc['Total Assets']
    int_coverage = ist.loc['EBIT'] / ist.loc['Interest Expense']
    debt_to_ebit = bs.loc['Total Debt'] / ist.loc['EBIT']

    return pd.DataFrame({
        'ticker': [ticker],
        'ebit_margin': [ebit_margin.iloc[0]],
        'cash_ratio': [cash_ratio.iloc[0]],
        'ROA': [ROA.iloc[0]],
        'int_coverage': [int_coverage.iloc[0]],
        'debt_to_ebit': [debt_to_ebit.iloc[0]]
    })

## <font color='cornflowerblue'> Ranges

In [5]:
ipc_tickers = ["AC.MX", "AMXB.MX", "ASURB.MX", "BIMBOA.MX", "CEMEXCPO.MX",
    "CUERVO.MX", "ELEKTRA.MX", "FEMSAUBD.MX", "GAPB.MX", "GCARSOA1.MX",
    "GCC.MX", "GMEXICOB.MX", "GRUMAB.MX", "KIMBERA.MX", "KOFUBL.MX",
    "LABB.MX", "LIVEPOLC-1.MX", "MEGACPO.MX", "OMAB.MX", "ORBIA.MX",
    "PE&OLES.MX", "PINFRA.MX", "SIGMAFA.MX", "TLEVISACPO.MX", "VESTA.MX",
    "WALMEX.MX"]

In [6]:
companies = []

for ticker in ipc_tickers:
    bs, ist, _ = get_financials(ticker)
    variables = model_variables(ticker, bs, ist)
    companies.append(variables)

df_companies = pd.concat(companies, ignore_index=True)

## <font color='cornflowerblue'> Methodologies for Credit Score

### <font color='skyblue'> Percentile

In [7]:
def score_variable(value, benchmark_series, inverse=False):
    p = percentileofscore(benchmark_series.dropna(), value, kind='rank')
    return (100 - p) if inverse else p

In [8]:
def total_score(row, df):
    ebit_margin_score = score_variable(row['ebit_margin'], df['ebit_margin'])
    cash_ratio_score = score_variable(row['cash_ratio'], df['cash_ratio'])
    ROA_score = score_variable(row['ROA'], df['ROA'])
    int_coverage_score = score_variable(row['int_coverage'], df['int_coverage'])
    debt_to_ebit_score = score_variable(row['debt_to_ebit'], df['debt_to_ebit'], inverse=True)

    return (ebit_margin_score + cash_ratio_score + ROA_score + int_coverage_score + debt_to_ebit_score) / 5

In [9]:
total_scores = df_companies.apply(lambda row: total_score(row, df_companies), axis=1)
df_companies['total_score'] = total_scores
df_companies.sort_values(by='total_score', ascending=False)

,ticker,ebit_margin,cash_ratio,ROA,int_coverage,debt_to_ebit,total_score
21,PINFRA.MX,1.134747,6.030469,0.153511,20.158093,0.003436,96.923077
11,GMEXICOB.MX,0.502216,3.361213,0.120100,14.510824,1.162752,89.230769
10,GCC.MX,0.297657,3.199208,0.083427,30.557353,1.560690,80.769231
20,PE&OLES.MX,0.297258,1.506765,0.104993,15.650337,1.335561,80.769231
2,ASURB.MX,0.442944,1.883110,0.117528,10.744204,2.158447,76.153846
18,OMAB.MX,0.570477,0.454670,0.172661,6.172917,1.491873,76.153846
5,CUERVO.MX,0.301738,0.869942,0.081486,11.384413,1.602108,73.846154
13,KIMBERA.MX,0.230198,0.620440,0.166139,6.055411,1.679835,68.461538
8,GAPB.MX,0.446395,0.628022,0.108522,4.229175,2.872289,63.076923
24,VESTA.MX,1.013579,4.104331,0.053254,5.406252,4.447187,60.769231


In [10]:
non_ipc_tickers = [t for t in all_tickers if t not in ipc_tickers]

non_ipc_companies = []

for ticker in non_ipc_tickers:
    bs, ist, _ = get_financials(ticker)
    variables = model_variables(ticker, bs, ist)
    non_ipc_companies.append(variables)

df_non_ipc_companies = pd.concat(non_ipc_companies, ignore_index=True)
df_non_ipc_companies = df_non_ipc_companies.fillna(0)

In [11]:
total_scores = df_non_ipc_companies.apply(lambda row: total_score(row, df_companies), axis=1)
df_non_ipc_companies['total_score'] = total_scores
df_non_ipc_companies.sort_values(by='total_score', ascending=False)

,ticker,ebit_margin,cash_ratio,ROA,int_coverage,debt_to_ebit,total_score
16,CIEB.MX,2.391000,1.304276,0.577809,79.743208,0.025707,94.615385
20,CULTIBAB.MX,13.265757,309.561656,0.095369,5631.530909,0.000000,93.076923
25,FRES.MX,0.470278,2.855912,0.190127,34.012749,0.396715,93.076923
17,CMOCTEZ.MX,0.431105,1.986122,0.303925,382.276260,0.041313,91.538462
53,SIMECB.MX,0.372704,2.858671,0.143084,3360.394321,0.000494,90.000000
...,...,...,...,...,...,...,...
59,TRAXIONA.MX,0.072361,0.164724,0.012761,1.366009,6.547053,10.000000
66,VOLARA.MX,0.063713,0.389282,-0.018428,0.620728,19.925897,9.230769
18,CONVERA.MX,0.047929,0.067513,-0.001933,0.863060,6.832623,6.153846
29,GISSAA.MX,0.002777,0.244861,-0.010865,0.136184,118.433539,5.384615


In [12]:
df_non_ipc_companies.total_score.mean(), df_companies.total_score.mean()

(36.39494833524684, 51.153846153846146)

### <font color='skyblue'> Weighted Average

In [13]:
weights = np.array([0.2, 0.1, 0.15, 0.3, 0.25])

def total_weighted_score(row, df, weights):
    ebit_margin_score = score_variable(row['ebit_margin'], df['ebit_margin'])
    cash_ratio_score = score_variable(row['cash_ratio'], df['cash_ratio'])
    ROA_score = score_variable(row['ROA'], df['ROA'])
    int_coverage_score = score_variable(row['int_coverage'], df['int_coverage'])
    debt_to_ebit_score = score_variable(row['debt_to_ebit'], df['debt_to_ebit'], inverse=True)

    return (weights[0] * ebit_margin_score + weights[1] * cash_ratio_score + weights[2] * ROA_score + weights[3] * int_coverage_score + weights[4] * debt_to_ebit_score)

In [14]:
df_companies['weighted_score'] = df_companies.apply(lambda row: total_weighted_score(row, df_companies, weights), axis=1)
df_companies.sort_values(by='weighted_score', ascending=False)

,ticker,ebit_margin,cash_ratio,ROA,int_coverage,debt_to_ebit,total_score,weighted_score
21,PINFRA.MX,1.134747,6.030469,0.153511,20.158093,0.003436,96.923077,96.730769
11,GMEXICOB.MX,0.502216,3.361213,0.120100,14.510824,1.162752,89.230769,88.846154
10,GCC.MX,0.297657,3.199208,0.083427,30.557353,1.560690,80.769231,82.500000
20,PE&OLES.MX,0.297258,1.506765,0.104993,15.650337,1.335561,80.769231,82.307692
18,OMAB.MX,0.570477,0.454670,0.172661,6.172917,1.491873,76.153846,76.730769
5,CUERVO.MX,0.301738,0.869942,0.081486,11.384413,1.602108,73.846154,75.576923
2,ASURB.MX,0.442944,1.883110,0.117528,10.744204,2.158447,76.153846,74.038462
13,KIMBERA.MX,0.230198,0.620440,0.166139,6.055411,1.679835,68.461538,66.538462
0,AC.MX,0.157824,0.462046,0.066491,9.075210,1.662782,59.230769,62.692308
8,GAPB.MX,0.446395,0.628022,0.108522,4.229175,2.872289,63.076923,59.038462


In [15]:
df_non_ipc_companies['weighted_score'] = df_non_ipc_companies.apply(lambda row: total_weighted_score(row, df_companies, weights), axis=1)
df_non_ipc_companies.sort_values(by='weighted_score', ascending=False)

,ticker,ebit_margin,cash_ratio,ROA,int_coverage,debt_to_ebit,total_score,weighted_score
16,CIEB.MX,2.391000,1.304276,0.577809,79.743208,0.025707,94.615385,96.730769
20,CULTIBAB.MX,13.265757,309.561656,0.095369,5631.530909,0.000000,93.076923,94.807692
25,FRES.MX,0.470278,2.855912,0.190127,34.012749,0.396715,93.076923,94.423077
17,CMOCTEZ.MX,0.431105,1.986122,0.303925,382.276260,0.041313,91.538462,92.884615
53,SIMECB.MX,0.372704,2.858671,0.143084,3360.394321,0.000494,90.000000,92.115385
...,...,...,...,...,...,...,...,...
59,TRAXIONA.MX,0.072361,0.164724,0.012761,1.366009,6.547053,10.000000,9.423077
18,CONVERA.MX,0.047929,0.067513,-0.001933,0.863060,6.832623,6.153846,6.538462
66,VOLARA.MX,0.063713,0.389282,-0.018428,0.620728,19.925897,9.230769,5.576923
29,GISSAA.MX,0.002777,0.244861,-0.010865,0.136184,118.433539,5.384615,3.269231


## <font color='cornflowerblue'> Payment Plans

In [16]:
inventory_turnover = ist.loc['Cost Of Revenue'] / bs.loc['Inventory']
inventory_days = 365 / inventory_turnover
inventory_days

2025-12-31    2.131875
dtype: object

In [17]:
receivables_turnover = ist.loc['Total Revenue'] / bs.loc['Accounts Receivable']
sales_days = 365 / receivables_turnover
sales_days

2025-12-31    7.598209
dtype: object